# Graph Neural Networks for Fraud Detection

This notebook demonstrates a comparative study of Graph Neural Networks (GNNs) versus traditional machine learning models for financial fraud detection.

## Overview

We will:
1. Load and explore transaction data
2. Build a transaction graph
3. Train GNN models (GCN, GAT, GraphSAGE)
4. Train baseline ML models (Logistic Regression, Random Forest, XGBoost)
5. Compare performance across all models

In [ ]:
# Imports
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path
sys.path.append('..')

from src.data import load_transaction_data, TransactionGraphBuilder
from src.data.data_loader import generate_synthetic_data
from src.models import GCNModel, GATModel, GraphSAGEModel, BaselineMLModels
from src.utils import GNNTrainer, ModelEvaluator
from src.utils.visualization import (
    plot_results, plot_comparison, plot_confusion_matrices, 
    plot_metrics_comparison, plot_graph
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. Load and Explore Data

In [ ]:
# Generate synthetic transaction data
# In practice, you would load real data using:
# df, labels = load_transaction_data('path/to/data.csv')

df, labels = generate_synthetic_data(
    n_transactions=5000,
    fraud_rate=0.1,
    n_users=300,
    n_merchants=150,
    random_state=42
)

print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few transactions:")
df.head()

In [ ]:
# Explore fraud distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud distribution
labels.value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Transaction Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)

# Amount distribution by fraud status
fraud_amounts = df[labels == 1]['amount']
legit_amounts = df[labels == 0]['amount']

axes[1].hist([legit_amounts, fraud_amounts], bins=30, label=['Legitimate', 'Fraud'], alpha=0.7)
axes[1].set_title('Transaction Amount Distribution')
axes[1].set_xlabel('Amount')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Build Transaction Graph

In [ ]:
# Build graph representation
builder = TransactionGraphBuilder()
graph_data = builder.build_graph(df, labels)

print(f"\nGraph statistics:")
print(f"  Number of nodes: {graph_data.num_nodes}")
print(f"  Number of edges: {graph_data.edge_index.shape[1]}")
print(f"  Node feature dimension: {graph_data.x.shape[1]}")
print(f"  Edge feature dimension: {graph_data.edge_attr.shape[1]}")
print(f"  Fraud edges: {graph_data.y.sum().item()} ({100*graph_data.y.float().mean():.2f}%)")

In [ ]:
# Visualize a subgraph
nx_graph = builder.to_networkx(graph_data)
edge_colors = ['red' if graph_data.y[i].item() == 1 else 'gray' 
               for i in range(graph_data.edge_index.shape[1])]

plot_graph(nx_graph, edge_colors=edge_colors, max_nodes=50)

## 3. Split Data

In [ ]:
from sklearn.model_selection import train_test_split

def split_data(graph_data, train_ratio=0.7, val_ratio=0.15, random_state=42):
    n_edges = graph_data.edge_index.shape[1]
    indices = np.arange(n_edges)
    
    # First split: train and temp (val + test)
    train_idx, temp_idx = train_test_split(
        indices,
        train_size=train_ratio,
        random_state=random_state,
        stratify=graph_data.y.numpy()
    )
    
    # Second split: val and test
    val_size = val_ratio / (1 - train_ratio)
    val_idx, test_idx = train_test_split(
        temp_idx,
        train_size=val_size,
        random_state=random_state,
        stratify=graph_data.y[temp_idx].numpy()
    )
    
    # Create masks
    train_mask = torch.zeros(n_edges, dtype=torch.bool)
    val_mask = torch.zeros(n_edges, dtype=torch.bool)
    test_mask = torch.zeros(n_edges, dtype=torch.bool)
    
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True
    
    return train_mask, val_mask, test_mask

train_mask, val_mask, test_mask = split_data(graph_data)

print(f"Train edges: {train_mask.sum().item()}")
print(f"Val edges: {val_mask.sum().item()}")
print(f"Test edges: {test_mask.sum().item()}")

## 4. Train GNN Models

In [ ]:
# Configuration
hidden_dim = 64
num_layers = 2
epochs = 100
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {device}")

in_channels = graph_data.x.shape[1]
edge_dim = graph_data.edge_attr.shape[1]

gnn_results = {}

### 4.1 GCN Model

In [ ]:
# Train GCN
gcn_model = GCNModel(in_channels, hidden_dim, num_layers)
gcn_trainer = GNNTrainer(gcn_model, device=device, learning_rate=0.01)

print("Training GCN...")
gcn_history = gcn_trainer.fit(
    graph_data, train_mask, val_mask,
    epochs=epochs,
    early_stopping_patience=20,
    verbose=True
)

# Evaluate
_, _, y_pred, y_proba = gcn_trainer.evaluate(graph_data, test_mask)
y_true = graph_data.y[test_mask].numpy()
y_proba_pos = y_proba[:, 1]

gcn_metrics = ModelEvaluator.evaluate(y_true, y_pred, y_proba_pos)
gnn_results['GCN'] = gcn_metrics
ModelEvaluator.print_evaluation(gcn_metrics, 'GCN')

plot_results(gcn_history)

### 4.2 GAT Model

In [ ]:
# Train GAT
gat_model = GATModel(in_channels, hidden_dim, num_layers, edge_dim=edge_dim)
gat_trainer = GNNTrainer(gat_model, device=device, learning_rate=0.01)

print("Training GAT...")
gat_history = gat_trainer.fit(
    graph_data, train_mask, val_mask,
    epochs=epochs,
    early_stopping_patience=20,
    verbose=True
)

# Evaluate
_, _, y_pred, y_proba = gat_trainer.evaluate(graph_data, test_mask)
y_true = graph_data.y[test_mask].numpy()
y_proba_pos = y_proba[:, 1]

gat_metrics = ModelEvaluator.evaluate(y_true, y_pred, y_proba_pos)
gnn_results['GAT'] = gat_metrics
ModelEvaluator.print_evaluation(gat_metrics, 'GAT')

plot_results(gat_history)

### 4.3 GraphSAGE Model

In [ ]:
# Train GraphSAGE
sage_model = GraphSAGEModel(in_channels, hidden_dim, num_layers)
sage_trainer = GNNTrainer(sage_model, device=device, learning_rate=0.01)

print("Training GraphSAGE...")
sage_history = sage_trainer.fit(
    graph_data, train_mask, val_mask,
    epochs=epochs,
    early_stopping_patience=20,
    verbose=True
)

# Evaluate
_, _, y_pred, y_proba = sage_trainer.evaluate(graph_data, test_mask)
y_true = graph_data.y[test_mask].numpy()
y_proba_pos = y_proba[:, 1]

sage_metrics = ModelEvaluator.evaluate(y_true, y_pred, y_proba_pos)
gnn_results['GraphSAGE'] = sage_metrics
ModelEvaluator.print_evaluation(sage_metrics, 'GraphSAGE')

plot_results(sage_history)

## 5. Train Baseline ML Models

In [ ]:
# Prepare features for baseline models
def prepare_baseline_features(df, graph_data):
    edge_features = graph_data.edge_attr.numpy()
    edge_index = graph_data.edge_index.numpy()
    node_features = graph_data.x.numpy()
    
    src_node_features = node_features[edge_index[0]]
    dst_node_features = node_features[edge_index[1]]
    
    combined_features = np.hstack([
        edge_features,
        src_node_features,
        dst_node_features
    ])
    
    return combined_features

X = prepare_baseline_features(df, graph_data)
y = graph_data.y.numpy()

X_train = X[train_mask.numpy()]
X_test = X[test_mask.numpy()]
y_train = y[train_mask.numpy()]
y_test = y[test_mask.numpy()]

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

In [ ]:
# Train baseline models
baseline = BaselineMLModels()
baseline.fit(X_train, y_train, model_name='all')

baseline_results = {}

for model_name in baseline.available_models():
    print(f"\nEvaluating {model_name}...")
    
    y_pred = baseline.predict(X_test, model_name)
    y_proba = baseline.predict_proba(X_test, model_name)[:, 1]
    
    metrics = ModelEvaluator.evaluate(y_test, y_pred, y_proba)
    baseline_results[model_name] = metrics
    
    ModelEvaluator.print_evaluation(metrics, model_name)

## 6. Compare All Models

In [ ]:
# Combine all results
all_results = {**gnn_results, **baseline_results}

# Create comparison table
comparison_df = ModelEvaluator.compare_models(all_results)
print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

In [ ]:
# Visualize comparisons
plot_comparison(comparison_df)

In [ ]:
# Plot confusion matrices
plot_confusion_matrices(all_results)

In [ ]:
# Radar chart comparison
plot_metrics_comparison(all_results)

## 7. Analysis and Conclusions

Based on the results above, we can draw several conclusions:

1. **GNN Performance**: Graph Neural Networks leverage the network structure of transactions, which can capture relational patterns that traditional models miss.

2. **Model Comparison**: 
   - GNNs typically excel when there are strong network effects (e.g., fraud rings)
   - Traditional ML models perform well when features are highly discriminative
   - The best model depends on the specific dataset characteristics

3. **Key Metrics**:
   - **Precision**: Important for minimizing false positives (legitimate transactions flagged as fraud)
   - **Recall**: Critical for catching actual fraud cases
   - **F1 Score**: Balanced measure, especially important for imbalanced datasets
   - **AUC-ROC**: Overall model discrimination ability

4. **Practical Considerations**:
   - GNNs require more computational resources
   - Traditional models are easier to interpret and deploy
   - Hybrid approaches may offer the best of both worlds